# Experiment 14A - Front View Stabilization

Notebook ini disinkronkan dengan preset `experiment_14A` di `src/config.py`.

Alur yang disarankan: jalankan training, evaluasi, dan fusion dari terminal terlebih dahulu. Setelah selesai, buka notebook ini untuk membaca ringkasan hasil dari file JSON yang sudah dibuat pipeline.

## Tujuan dan Hipotesis

Experiment 14A dilakukan karena front view pada Experiment 13 masih menunjukkan generalization gap, sedangkan side view sudah relatif stabil. Experiment 14A awal berhasil mengecilkan gap, tetapi collapse ke kelas mayoritas `phone_use` dan hampir tidak mengenali `safe_driving`.

Revisi 14A mengembalikan freeze stage ke 5 dan mematikan label smoothing. Hipotesis: model kembali mampu mengenali `safe_driving`, sementara dropout 0.4 dan weight decay 3e-4 tetap memberi regularisasi ringan.

## Konfigurasi Experiment 14A

- View: front
- Backbone: EfficientNetV2-S
- Optimizer: AdamW
- Learning rate: 3e-5
- Weight decay: 3e-4
- Dropout: 0.4
- Label smoothing: 0.0
- Freeze stage: 5
- Scheduler: ReduceLROnPlateau
- Scheduler factor: 0.5
- Scheduler patience: 1
- Early stopping: aktif, patience 4
- Batch size: 32
- Max epoch: 30
- Best checkpoint: validation Macro F1
- Side view: tetap memakai checkpoint Experiment 13 sebagai kontrol

## Command Terminal

Jalankan dari root repository `D:\\Skripsi\\Experiment`.

```powershell
Copy-Item checkpoints\side_best.pt checkpoints\side_best_exp13_backup.pt
python -m src.train --view front --experiment experiment_14A
python -m src.evaluate --view front --checkpoint checkpoints\front_best_exp14A.pt
python -m src.fusion --front-checkpoint checkpoints\front_best_exp14A.pt --side-checkpoint checkpoints\side_best_exp13_backup.pt
```

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RESULTS_DIR = PROJECT_ROOT / 'results'
CHECKPOINT_DIR = PROJECT_ROOT / 'checkpoints'

history_path = RESULTS_DIR / 'front_history_exp14A.json'
metrics_path = RESULTS_DIR / 'experiment_14A_front_metrics.json'
fusion_path = RESULTS_DIR / 'fusion_comparison_exp14A.json'

print('Project root:', PROJECT_ROOT)
print('History exists:', history_path.exists(), history_path)
print('Metrics exists:', metrics_path.exists(), metrics_path)
print('Fusion exists:', fusion_path.exists(), fusion_path)

In [ ]:
if history_path.exists():
    history = json.loads(history_path.read_text())
    hist_df = pd.DataFrame(history)
    hist_df.index = hist_df.index + 1
    hist_df.index.name = 'epoch'
    display(hist_df)
else:
    print('History belum ada. Jalankan training front Experiment 14A dari terminal terlebih dahulu.')

In [ ]:
if history_path.exists():
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    hist_df[['train_loss', 'val_loss']].plot(ax=axes[0], marker='o')
    axes[0].set_title('Loss Curve')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].grid(True, alpha=0.3)

    cols = [c for c in ['val_macro_f1', 'train_val_loss_gap'] if c in hist_df.columns]
    hist_df[cols].plot(ax=axes[1], marker='o')
    axes[1].set_title('Validation F1 and Loss Gap')
    axes[1].set_xlabel('Epoch')
    axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
else:
    print('Belum ada history untuk divisualisasikan.')

In [ ]:
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text())
    summary_keys = [
        'best_epoch',
        'val_macro_f1',
        'train_loss_at_best_epoch',
        'validation_loss_at_best_epoch',
        'train_validation_loss_gap_at_best_epoch',
        'test_accuracy',
        'test_precision_macro',
        'test_recall_macro',
        'test_macro_f1',
        'ece_binary',
        'brier_score',
    ]
    display(pd.Series({k: metrics.get(k) for k in summary_keys}, name='experiment_14A_front'))
    display(pd.DataFrame(metrics.get('confusion_matrix', []), index=['true_safe', 'true_phone'], columns=['pred_safe', 'pred_phone']))
else:
    print('Metrics belum ada. Jalankan evaluasi front Experiment 14A dari terminal terlebih dahulu.')

In [ ]:
if fusion_path.exists():
    fusion = json.loads(fusion_path.read_text())
    rows = []
    for name, item in fusion.items():
        rows.append({
            'method': name,
            'accuracy': item.get('accuracy'),
            'precision_macro': item.get('precision_macro'),
            'recall_macro': item.get('recall_macro'),
            'f1_macro': item.get('f1_macro'),
        })
    display(pd.DataFrame(rows).set_index('method'))
else:
    print('Fusion comparison belum ada. Jalankan fusion dari terminal setelah evaluasi selesai.')

## Interpretasi Singkat

- Jika train loss dan validation loss sama-sama tinggi, regularisasi terlalu kuat atau freeze terlalu banyak.
- Jika train loss rendah tetapi validation loss tetap tinggi, overfit masih terjadi.
- Jika Macro F1 naik tetapi validation loss memburuk, model mungkin makin overconfident.
- Jika gap mengecil dan Macro F1 stabil atau turun sedikit, Experiment 14A masih dapat dianggap berhasil sebagai stabilisasi.